# Value analysis of the removed duplicate rows

Check whether the values of rows removed by canonical SMILES are usable for training.

In [4]:
import pandas as pd
import os
import numpy as np
from rdkit import Chem

data_dir = r'../data/in_vitro_data'
output_dir = r'../data/in_vitro_data_deduplicated'

files = [
    'chembl_CL_raw.csv',
    'chembl_Fu_raw.csv',
    'biogen_CL_raw.csv',
    'chembl_PPB_raw.csv',
    'tdc_cl_hep.csv',
    'tdc_cl_mic.csv',
    'tdc_half_life.csv',
    'tdc_PPBR_AZ.csv'
]

smiles_column_candidates = ['smiles', 'SMILES', 'Smiles', 'Drug']
value_column_candidates = ['value', 'Y']

def get_canonical_smiles(smiles_str):
    if pd.isna(smiles_str):
        return None
    try:
        mol = Chem.MolFromSmiles(str(smiles_str))
        if mol is not None:
            return Chem.MolToSmiles(mol)
        else:
            return None
    except:
        return None

print("Analyzing removed duplicates...\n")
print("="*100)

Analyzing removed duplicates...



In [5]:
for file in files:
    file_path = os.path.join(data_dir, file)
    
    try:
        df = pd.read_csv(file_path)
        
        # Find the SMILES column
        smiles_col = None
        for col in smiles_column_candidates:
            if col in df.columns:
                smiles_col = col
                break
        
        # Find the value column
        value_col = None
        for col in value_column_candidates:
            if col in df.columns:
                value_col = col
                break
        
        if not smiles_col or not value_col:
            print(f"✗ {file}: Missing SMILES or value column\n")
            continue
        
        # Canonical SMILES conversion
        df['canonical_smiles'] = df[smiles_col].apply(get_canonical_smiles)
        df_valid = df[df['canonical_smiles'].notna()].copy()
        
        # Duplicated rows among the valid rows
        df_duplicates = df_valid[df_valid['canonical_smiles'].duplicated(keep=False)]
        
        # Rows to be removed from each canonical SMILES group (all but the first)
        df_to_remove = df_duplicates[df_duplicates.duplicated(subset=['canonical_smiles'], keep='first')]
        
        if len(df_to_remove) > 0:
            print(f"📌 {file}")
            print(f"   Removed rows: {len(df_to_remove)}\n")
            
            # Value analysis
            removed_values = pd.to_numeric(df_to_remove[value_col], errors='coerce')
            kept_values = pd.to_numeric(df_duplicates[df_duplicates.duplicated(subset=['canonical_smiles'], keep='first') == False][value_col], errors='coerce')
            
            print(f"   Value statistics of removed rows:")
            print(f"      Count: {removed_values.notna().sum()}")
            print(f"      Mean: {removed_values.mean():.4f}")
            print(f"      Std: {removed_values.std():.4f}")
            print(f"      Min: {removed_values.min():.4f}")
            print(f"      Max: {removed_values.max():.4f}")
            print(f"      Median: {removed_values.median():.4f}")
            print()
            
            print(f"   Value statistics of kept rows (first occurrence):")
            print(f"      Count: {kept_values.notna().sum()}")
            print(f"      Mean: {kept_values.mean():.4f}")
            print(f"      Std: {kept_values.std():.4f}")
            print(f"      Min: {kept_values.min():.4f}")
            print(f"      Max: {kept_values.max():.4f}")
            print(f"      Median: {kept_values.median():.4f}")
            print()
            
            # doc_id and assay_id analysis
            print(f"   doc_id/assay_id analysis for the same molecule:")
            
            # Duplicate analysis per molecule
            same_doc_assay = 0  # same doc_id + assay_id
            diff_doc_or_assay = 0  # different doc_id or assay_id
            
            for smiles, group in df_duplicates.groupby('canonical_smiles'):
                if len(group) > 1:
                    for i in range(1, len(group)):
                        current = group.iloc[i]
                        first = group.iloc[0]
                        
                        if 'doc_id' in group.columns and 'assay_id' in group.columns:
                            if (current['doc_id'] == first['doc_id'] and 
                                current['assay_id'] == first['assay_id']):
                                same_doc_assay += 1
                            else:
                                diff_doc_or_assay += 1
            
            total_dups = same_doc_assay + diff_doc_or_assay
            if total_dups > 0:
                print(f"      Same paper + same assay: {same_doc_assay} ({same_doc_assay/total_dups*100:.1f}%) - true duplicates")
                print(f"      Different paper or assay: {diff_doc_or_assay} ({diff_doc_or_assay/total_dups*100:.1f}%) - independent measurements")
            print()
            
            # Examples of the same molecule with different values (with doc_id/assay_id)
            print(f"   Duplicate examples for the same molecule (top 3):")
            count = 0
            for smiles, group in df_duplicates.groupby('canonical_smiles'):
                if len(group) > 1:
                    values = group[value_col].values
                    if len(set(values)) > 1:  # only when the values differ
                        print(f"      Molecule {count+1}: {smiles[:50]}...")
                        for idx, (_, row) in enumerate(group.iterrows()):
                            doc_id = row['doc_id'] if 'doc_id' in row else 'N/A'
                            assay_id = row['assay_id'] if 'assay_id' in row else 'N/A'
                            print(f"         Row {idx+1}: {value_col}={row[value_col]}, doc_id={doc_id}, assay_id={assay_id}")
                        count += 1
                        if count >= 3:
                            break
            print("\n" + "="*100 + "\n")
        else:
            print(f"✓ {file}: No duplicates to remove\n")
            
    except Exception as e:
        print(f"✗ {file}: Error - {e}\n")

📌 chembl_CL_raw.csv
   Removed rows: 36169

   Value statistics of removed rows:
      Count: 36169
      Mean: 4699.5245
      Std: 331581.1181
      Min: -46.6000
      Max: 59000000.0000
      Median: 21.0000

   Value statistics of kept rows (first occurrence):
      Count: 18143
      Mean: 1659.9345
      Std: 36869.9699
      Min: -56.9000
      Max: 2345000.0000
      Median: 23.0000

   doc_id/assay_id analysis for the same molecule:
      Same paper + same assay: 375 (1.0%) - true duplicates
      Different paper or assay: 35794 (99.0%) - independent measurements

   Duplicate examples for the same molecule (top 3):
      Molecule 1: Br.COc1ccc2c3c1O[C@H]1C[C@@H](O)C=C[C@@]31CCN(C)C2...
         Row 1: value=5.667, doc_id=CHEMBL3526005, assay_id=CHEMBL3540922
         Row 2: value=34.0, doc_id=CHEMBL3526005, assay_id=CHEMBL3540300
         Row 3: value=41.0, doc_id=CHEMBL3526005, assay_id=CHEMBL3540720
      Molecule 2: Br.Cc1cnc(Nc2ccc(C#N)cc2)nc1Nc1c(F)cc(-c2ccncc2)cc...
  

## Summary and recommendations

In [ ]:
print("\n" + "="*100)
print("Summary and recommendations")
print("="*100)
print("""
Interpretation of the analysis:

1. "Same paper + same assay" (true duplicates):
   - identical doc_id and assay_id
   - most likely repeats of the same experiment
   - recommended: merge by mean/median
   - or keep only the single most reliable value

2. "Different paper or different assay" (independent measurements):
   - a different paper or different experimental conditions
   - the same molecule measured under various conditions
   - valuable data - recommended: merge by mean or keep all
   - helps model generalisation

3. Action plan:
   ✓ ChEMBL data with doc_id/assay_id:
     → merge only true duplicates by mean, keep all independent measurements
   
   ✓ TDC data without doc_id/assay_id:
     → keep only the first as now (no selection criterion)
     or merge by mean (safe, since the distributions are similar)
""")